# Training on INCLUDE_50 Dataset

## Preprocessing Data

In [1]:
import pandas as pd
import numpy as np
import os

import tensorflow as tf
from tensorflow.keras.utils import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Input, Dense, LSTM, Bidirectional, Flatten

from sklearn.model_selection import train_test_split

from tqdm.notebook import tqdm

In [2]:
# Checking whether the gpu is available
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        print("Using GPU:", gpus)

    except RuntimeError as e:
        print(e)

else:
    print("No GPU found.")

Using GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
train_data = pd.read_csv('Dataset/train.csv')
test_data = pd.read_csv('Dataset/test.csv')

train_data.sample(5)

,parent_label,label,video_path,include_50
2832,Places,26. University,Places/26. University/MVI_3622.mp4,False
1378,Home,46. Box,Home/46. Box/MVI_4945.mp4,False
3679,Greetings,51. Good Morning,Greetings/51. Good Morning/MVI_9991.mp4,True
1658,Adjectives,84. small little,Adjectives/84. small little/MVI_5287.mp4,False
241,Greetings,54. Good night,Greetings/54. Good night/MVI_0110.mp4,False


In [25]:
# labels = train_data['label']
# train_data_path = train_data['vedio_path']

# Extracting labels and video paths for the videos that are from the include 50 dataset

labels = []
train_path_I50 = []

for i in range(len(train_data)):
    if train_data['include_50'][i] == True:
        labels.append(train_data['label'][i])
        train_path_I50.append("Dataset\\" + train_data['video_path'][i])
        
labels = pd.Series(labels).unique()
labels = pd.Series(labels).to_list()

train_path_I50 = pd.Series(train_path_I50)

# train_path_I50.sample(5), labels.sample(5)
labels[:5]

['1. loud', '19. House', '83. big large', '91. Priest', '23. Court']

In [26]:
label_map = dict()

# Creating a dictionary of labels with their corresponding index 
for i in range(len(labels)):
    # Splitting Label from num
    split = labels[i].split(" ")
    
    # Joining the ramining str to make a full label name
    label = split[1:]
    label = " ".join(label)
        
    label_map[i] = label

labels = list(label_map.values())
  
label_map    

{0: 'loud',
 1: 'House',
 2: 'big large',
 3: 'Priest',
 4: 'Court',
 5: 'train ticket',
 6: 'it',
 7: 'Shoes',
 8: 'Dog',
 9: 'Bank',
 10: 'Thank you',
 11: 'Election',
 12: 'Cow',
 13: 'Window',
 14: 'quiet',
 15: 'dry',
 16: 'long',
 17: 'Hello',
 18: 'Bird',
 19: 'Hat',
 20: 'Black',
 21: 'short',
 22: 'White',
 23: 'Fan',
 24: 'new',
 25: 'Store or Shop',
 26: 'Monday',
 27: 'Death',
 28: 'Cell phone',
 29: 'you (plural)',
 30: 'T-Shirt',
 31: 'Girl',
 32: 'Father',
 33: 'Red',
 34: 'hot',
 35: 'Fall',
 36: 'I',
 37: 'Time',
 38: 'Car',
 39: 'Good Morning',
 40: 'Summer',
 41: 'Paint',
 42: 'Teacher',
 43: 'Brother',
 44: 'good',
 45: 'happy',
 46: 'Boy',
 47: 'small little',
 48: 'Pen',
 49: 'Year'}

In [6]:
# Loading all the labeled videos in the dataset
X = []
y= []

for label in tqdm(labels):
    label_videos = os.listdir("MP_data/"+label)
    # window = []
    
    for video in label_videos:        
        res = np.load("MP_data/" + f"{label}/" + video,allow_pickle=True)
        X.append(res)
        y.append(label_map[label])        

len(X),len(y)

        

  0%|          | 0/50 [00:00<?, ?it/s]

(675, 675)

In [7]:
# X,y = np.array(X),np.array(y)
# X.shape,y.shape
len(X[0])

66

In [8]:
# Bhai yha meine kya hi harkat kari thi gandi wali bhai agar meine har vedio ka input ek baar mein hi pahucha dunga toh kaise kaam banega sab vedio ke inputs ko alag lena hoga na bhai :))))))

# Padding the videos to make them of the same length

max_frames = max([len(video) for video in X])
max_frames

X = pad_sequences(X, maxlen=max_frames, padding='post', dtype='float32')

print(X.shape)


(675, 154, 1662)


In [9]:
X= np.array(X)
y= np.array(y)
y = to_categorical(y)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42,shuffle=True) 

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)


X_train shape: (540, 154, 1662)
X_val shape: (135, 154, 1662)
y_train shape: (540, 50)
y_val shape: (135, 50)


# Model

## Architecture

In [ ]:
import keras

# input_shape = (154, 1662)
input_shape = X_train[0].shape #(154, 1662)
num_classes =  len(label_map.keys())#50

INCLUDESEQ50_V2 = keras.Sequential([        
        Input(shape=input_shape),        
        
        # Bidirectional LSTM layers
        Bidirectional(LSTM(64, return_sequences=True)),
        Bidirectional(LSTM(64, return_sequences=True)),
        Bidirectional(LSTM(64, return_sequences=True)),
        Bidirectional(LSTM(64, return_sequences=True)),
        
        # Flatten the output
        Flatten(),
        
        # Fully connected layer
        Dense(128, activation='relu'),
        
        # Output layer
        Dense(num_classes, activation='softmax')
])

model = INCLUDESEQ50_V2

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
 
model.build(input_shape=(input_shape))

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 bidirectional (Bidirectiona  (None, 154, 128)         884224    
 l)                                                              
                                                                 
 bidirectional_1 (Bidirectio  (None, 154, 128)         98816     
 nal)                                                            
                                                                 
 bidirectional_2 (Bidirectio  (None, 154, 128)         98816     
 nal)                                                            
                                                                 
 bidirectional_3 (Bidirectio  (None, 154, 128)         98816     
 nal)                                                            
                                                                 
 flatten (Flatten)           (None, 19712)             0

In [11]:
model.fit(X_train,
          y_train,
          validation_data=(X_val, y_val), 
          epochs=65)
# ,        #   batch_size=batch_size)


Epoch 1/65
17/17 [==============================] - 15s 262ms/step - loss: 4.0130 - accuracy: 0.0259 - val_loss: 3.8264 - val_accuracy: 0.0444
Epoch 2/65
17/17 [==============================] - 2s 111ms/step - loss: 3.8551 - accuracy: 0.0463 - val_loss: 3.8615 - val_accuracy: 0.0296
Epoch 3/65
17/17 [==============================] - 2s 111ms/step - loss: 3.7297 - accuracy: 0.0556 - val_loss: 3.7624 - val_accuracy: 0.0296
Epoch 4/65
17/17 [==============================] - 2s 113ms/step - loss: 3.6889 - accuracy: 0.0593 - val_loss: 3.7737 - val_accuracy: 0.0296
Epoch 5/65
17/17 [==============================] - 2s 109ms/step - loss: 3.6298 - accuracy: 0.0704 - val_loss: 3.7618 - val_accuracy: 0.0444
Epoch 6/65
17/17 [==============================] - 2s 111ms/step - loss: 3.5828 - accuracy: 0.0574 - val_loss: 3.7645 - val_accuracy: 0.0444
Epoch 7/65
17/17 [==============================] - 2s 117ms/step - loss: 3.5245 - accuracy: 0.0704 - val_loss: 3.8880 - val_accuracy: 0.0296
Epoch

In [28]:
# Predict probabilities for the test data
probabilities = model.predict(X_val)

# Convert probabilities to class labels
predicted_classes = np.argmax(probabilities, axis=-1)

for i in range(10):
    print(f'Predicted: {label_map[predicted_classes[i]]}, True: {label_map[np.argmax(y_val[i])]}')


5/5 [==============================] - 0s 46ms/step
Predicted: Red, True: you (plural)
Predicted: happy, True: short
Predicted: Court, True: Year
Predicted: quiet, True: Good Morning
Predicted: Boy, True: Death
Predicted: Priest, True: Priest
Predicted: Shoes, True: small little
Predicted: Car, True: Bird
Predicted: dry, True: Pen
Predicted: Brother, True: hot


In [29]:
model.save('INCLUDESEQ50_V2.h5')

1.640552e-10